# Generate Strategic Reports

## Load useful libraries

In [ ]:
import os
import shutil
from pathlib import Path
import json
import time
import papermill as pm

In [ ]:
from strategic_reports.daily.config.topic_order import list_directories_and_titles

## User settings

In [ ]:
directory_output_root = os.environ['STRATEGIC_REPORTS_HOME'] + '/output/daily'
directory_data_root = os.environ['STRATEGIC_REPORTS_HOME'] + '/data/rss_feeds'
directory_exe_root = os.environ['STRATEGIC_REPORTS_HOME'] + '/strategic_reports/daily'

subdirectory_name_report = 'strategic-report'
subdirectory_name_logs = 'logs'

report_header_level = '#'

number_of_seconds_to_sleep = 5

debug = False
debug_group = None

## Initialize useful variables

In [ ]:
directory_notebooks = directory_exe_root + '/notebooks'

## Create or replace output directory

In [ ]:
shutil.rmtree(directory_output_root, ignore_errors = True)

In [ ]:
Path(directory_output_root).mkdir()

In [ ]:
Path(directory_output_root + '/' + subdirectory_name_logs).mkdir()

## Set debugging parameters

In [ ]:
if debug:
    list_directories_and_titles = [x for x in list_directories_and_titles if x['slug'] in debug_group]

In [ ]:
if debug:
    print(list_directories_and_titles)

## Obtain the report results for each file

In [ ]:
list_failed_reports = []

for item in list_directories_and_titles:

    subdirectory = item['slug']

    print(subdirectory)

    parameters_pipeline = {
        'directory_output_root' : directory_output_root,
        'filename_json_feeds_list' : directory_data_root + '/' + subdirectory + '.json',
        'directory_notebooks' : directory_notebooks + '/per_topic',
    }

    try:
        result = pm.execute_notebook(
           directory_notebooks + '/per_topic/pipeline-per-topic.ipynb',
           directory_output_root + '/' + subdirectory_name_logs + '/OUTPUT-pipeline-per-topic-' + subdirectory.upper() + '.ipynb',
           parameters = parameters_pipeline,
        )
        time.sleep(number_of_seconds_to_sleep)
    except:
        list_failed_reports.append(subdirectory)
        continue

## Log pipeline failures

In [ ]:
with open(directory_output_root + '/' + subdirectory_name_logs + '/pipeline_failures.json', 'w') as f:
    f.write(json.dumps({'pipeline_failures' : list_failed_reports}) + '\n')

##### Log RSS feed pull exceptions

In [ ]:
list_all_exceptions = []
for item in list_directories_and_titles:
    subdirectory = item['slug']

    list_exceptions = []
   
    with open(directory_output_root + '/' + subdirectory + '/exceptions_feed_pull.json') as f:
        list_exceptions = json.load(f)['exceptions']

    list_all_exceptions.append({subdirectory : list_exceptions})

In [ ]:
with open(directory_output_root + '/' + subdirectory_name_logs + '/exceptions.json', 'w') as f:
    f.write(json.dumps({'exceptions' : list_failed_reports}) + '\n')

## Build report

(Note:  This does not include the "ner.ipynb" tags, just the LLM-generated tags).

In [ ]:
parameters_report = {
    'directory_output_root' : directory_output_root,
    'subdirectory_name_report' : subdirectory_name_report,
    'subdirectory_name_logs' : subdirectory_name_logs,
    'report_header_level' : report_header_level,
    'debug' : debug,
    'debug_group' : debug_group,
}

In [ ]:
result = pm.execute_notebook(
    directory_notebooks + '/assemble_results/create-publication.ipynb',
    directory_output_root + '/' + subdirectory_name_logs + '/OUTPUT-create-publication.ipynb',
    parameters = parameters_report,
)

In [ ]:
time.sleep(number_of_seconds_to_sleep)